# Nigeria Climate Data Analysis
Exploratory data analysis for Nigeria climate dataset.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load the Nigeria climate data
df = pd.read_csv('../nigeria.csv')

# Parse dates - YEAR and DOY (Day of Year) to datetime
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3), format='%Y%j')

# Add Country column
df['Country'] = 'Nigeria'

# Extract Month from DATE
df['Month'] = df['DATE'].dt.month

# Replace -999 with NaN
df = df.replace(-999, np.nan)

print("Data Loading Complete:")
print(f"- Dataset shape: {df.shape}")
print(f"- Date range: {df['DATE'].min()} to {df['DATE'].max()}")
print(f"- Country: {df['Country'].unique()}")

# Missing values analysis
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100
high_missing = missing_percentages[missing_percentages > 5]

print(f"\nColumns with >5% missing values: {len(high_missing)}")
if len(high_missing) > 0:
    print(high_missing)

# Data cleaning
df_clean = df.copy()
df_clean = df_clean.fillna(method='ffill')
missing_per_row = df.isnull().sum(axis=1) / len(df.columns)
rows_to_drop = missing_per_row > 0.3
df_clean = df_clean[~rows_to_drop]

print(f"\nDataset shape after cleaning: {df_clean.shape}")

# Export cleaned data
if not os.path.exists('../data'):
    os.makedirs('../data')
df_clean.to_csv('../data/nigeria_clean.csv', index=False)
print("Cleaned data exported to ../data/nigeria_clean.csv")

df = df_clean.copy()
df.head()

In [ ]:
# Time Series Analysis
monthly_temp = df.groupby('Month')['T2M'].mean()
monthly_precip = df.groupby('Month')['PRECTOTCORR'].sum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Temperature plot
ax1.plot(monthly_temp.index, monthly_temp.values, marker='o', linewidth=2, markersize=8)
ax1.set_title('Nigeria: Monthly Average Temperature (2015-2026)', fontweight='bold')
ax1.set_ylabel('Temperature (°C)')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, 13))

warmest_month = monthly_temp.idxmax()
coolest_month = monthly_temp.idxmin()
ax1.annotate(f'Warmest: {warmest_month} ({monthly_temp.max():.1f}°C)', 
             xy=(warmest_month, monthly_temp.max()), 
             xytext=(warmest_month+0.5, monthly_temp.max()+1),
             arrowprops=dict(arrowstyle='->', color='red'),
             color='red')

# Precipitation plot
ax2.bar(monthly_precip.index, monthly_precip.values, color='skyblue', alpha=0.7)
ax2.set_title('Nigeria: Monthly Total Precipitation (2015-2026)', fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Precipitation (mm)')
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xticks(range(1, 13))

peak_month = monthly_precip.idxmax()
ax2.annotate(f'Peak: {peak_month} ({monthly_precip.max():.1f}mm)', 
             xy=(peak_month, monthly_precip.max()), 
             xytext=(peak_month+0.5, monthly_precip.max()+20),
             arrowprops=dict(arrowstyle='->', color='darkblue'),
             color='darkblue')

plt.tight_layout()
plt.show()

print(f"Temperature range: {monthly_temp.max():.1f}°C to {monthly_temp.min():.1f}°C")
print(f"Peak rainfall: Month {peak_month} ({monthly_precip.max():.1f}mm)")

In [ ]:
# Correlation Analysis
numerical_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'PS']
correlation_data = df[numerical_vars].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_data, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Nigeria: Climate Variables Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations
corr_pairs = []
for i in range(len(correlation_data.columns)):
    for j in range(i+1, len(correlation_data.columns)):
        var1 = correlation_data.columns[i]
        var2 = correlation_data.columns[j]
        corr_val = correlation_data.iloc[i, j]
        if abs(corr_val) > 0.3:
            corr_pairs.append((var1, var2, corr_val))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("Top correlations:")
for i, (var1, var2, corr) in enumerate(corr_pairs[:3]):
    print(f"{i+1}. {var1} vs {var2}: {corr:.3f}")

In [ ]:
# Distribution Analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Precipitation distribution
precip_data = df['PRECTOTCORR'].dropna()
if precip_data.skew() > 1:
    ax1.hist(precip_data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax1.set_yscale('log')
    ax1.set_title('Precipitation Distribution (Log Scale)')
else:
    ax1.hist(precip_data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax1.set_title('Precipitation Distribution')

ax1.set_xlabel('Precipitation (mm)')
ax1.set_ylabel('Frequency')
ax1.grid(True, alpha=0.3)

# Bubble chart
sample_data = df.sample(n=min(1000, len(df)))
bubble_size = sample_data['WS2M'] * 10

scatter = ax2.scatter(sample_data['T2M'], sample_data['PRECTOTCORR'], 
                     s=bubble_size, alpha=0.3, c=sample_data['RH2M'], 
                     cmap='viridis', edgecolors='black', linewidth=0.5)

ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Precipitation (mm)')
ax2.set_title('Temperature vs Precipitation\n(Bubble: Wind Speed, Color: Humidity)')
plt.colorbar(scatter, ax=ax2, label='Relative Humidity (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Precipitation skewness: {precip_data.skew():.2f}")
print("\nNigeria shows tropical climate patterns with distinct wet and dry seasons")